# Step 13: Advanced Orchestration

        _Instructor Solution Notebook_  
        Estimated time: **90 minutes**  
        Difficulty: **Advanced**

        ## Learning Objectives
        - [ ] Implement a lightweight group-chat loop.
- [ ] Model manager-worker decomposition with parallel subtasks.
- [ ] Reason about orchestration state beyond one direct prompt-response pair.
- [ ] Prepare for longer-running systems and supervision.

        ## Prerequisites
        - Step 12 completed.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Summary & Key Takeaways
8. Additional Resources


## Setup & Imports

The next cell adds the repository root to `sys.path` and imports the shared course helpers.
Most notebooks use the same bootstrap so the lesson content can stay focused on the concept,
not on path setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

Advanced orchestration becomes easier to understand when we build it in plain Python first. This notebook keeps the control loop explicit so the role of each agent stays visible.

Expected output and validation notes:

Expected output snapshot:

- Group chat returns a small conversation transcript
- Manager-worker orchestration returns task and result lists


## Part 2: Core Implementation

### Group chat loop


In [ ]:
import asyncio

class GroupChat:
    def __init__(self, agents, max_turns: int = 4):
        self.agents = agents
        self.max_turns = max_turns
        self.history = []

    async def run(self, topic: str) -> list[str]:
        self.history = [f"Topic: {topic}"]
        current_prompt = topic
        for turn in range(self.max_turns):
            agent = self.agents[turn % len(self.agents)]
            response = await agent.run(
                "Conversation so far:\n" + "\n".join(self.history) + f"\n\nRespond to: {current_prompt}"
            )
            self.history.append(f"{agent.name}: {response.text}")
            current_prompt = response.text
        return self.history

group_chat = GroupChat(
    [
        create_agent(name="PM", instructions="Speak as a product manager."),
        create_agent(name="Engineer", instructions="Speak as an engineer."),
        create_agent(name="Designer", instructions="Speak as a designer."),
    ]
)


## Part 2: Core Implementation

### Manager-worker decomposition


In [ ]:
class ManagerWorkerOrchestrator:
    def __init__(self):
        self.manager = create_agent(
            name="Manager",
            instructions="Break a goal into 3 short, concrete subtasks."
        )
        self.workers = [
            create_agent(name="WorkerA", instructions="Complete the assigned subtask."),
            create_agent(name="WorkerB", instructions="Complete the assigned subtask."),
            create_agent(name="WorkerC", instructions="Complete the assigned subtask."),
        ]

    async def run(self, goal: str) -> dict:
        task_plan = await self.manager.run(f"Break this into 3 numbered tasks:\n{goal}")
        tasks = [line.strip() for line in task_plan.text.splitlines() if line.strip()[:1].isdigit()]
        results = await asyncio.gather(
            *(worker.run(task) for worker, task in zip(self.workers, tasks, strict=False))
        )
        return {
            "tasks": tasks,
            "results": [item.text for item in results],
        }

orchestrator = ManagerWorkerOrchestrator()


## Part 3: Hands-On Exercises

Run one group-chat topic and one manager-worker topic, then inspect the task breakdown or history.

This mirrored notebook uses completed exercise code so instructors can demonstrate the target state.


In [ ]:
history = await group_chat.run("Design a helpful onboarding flow for new agent-framework learners.")
print_json(history, title="Group chat history")

result = await orchestrator.run("Create a launch plan for an internal agent-learning workshop.")
print_json(result, title="Manager-worker result")


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
history = await group_chat.run("Design a helpful onboarding flow for new agent-framework learners.")
print_json(history, title="Group chat history")

result = await orchestrator.run("Create a launch plan for an internal agent-learning workshop.")
print_json(result, title="Manager-worker result")


## Part 5: Best Practices & Tips

        - Keep orchestration loops explicit while you are still learning the pattern.
- Distinguish role prompts from control logic.
- Inspect intermediate state instead of only the final synthesis.


## Summary & Key Takeaways

You now have two advanced orchestration patterns that make coordination state visible rather than magical.


## Additional Resources

        - `Step 12 notebook`
- `docs/references.md`
